In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [356]:
H_P1 = pd.read_csv("Original_Data//HarrisPartI.csv")
H_P2 = pd.read_csv("Original_Data//HarrisPartII.csv")
H_P3 = pd.read_csv("Original_Data//HarrisPartIII.csv")
V_T2 = pd.read_csv("Original_Data//vandenBerg_table2.csv")
K_21 = pd.read_csv("Original_Data//Krause21.csv")

DL=[H_P1,H_P2,H_P3,V_T2,K_21] # "DataList" (List of DataFrames) to allow for iteration.

# DL[3] is the vandenBerg data. Here we need to convert the NGC number into an ID.
DL[3].loc[DL[3]["#NGC"]!="XXXX","#NGC"] = "NGC" + DL[3].loc[DL[3]["#NGC"]!="XXXX","#NGC"]   # If the #NGC number exists (eg:1234), we change it to the format "NGC1234"
DL[3].loc[DL[3]["#NGC"]=="XXXX","#NGC"] =         DL[3].loc[DL[3]["#NGC"]=="XXXX","Name"]   # If the #NGC number dosent exist, we sub in its "Name"
DL[3].loc[DL[3]["#NGC"]==DL[3]["Name"],"Name"] =  "XXXX"                                    # If we subbed in the "Name" for a missing NGC, we cut out the name to avoid double counting later.
DL[3].rename(columns={'#NGC': 'ID'}, inplace=True)                                          # Renames the column that now stores our IDs

# DL[4] is the Krause data. Here we just need to rename some columns.
DL[4].rename(columns={'Object': 'ID'}, inplace=True)
DL[4].rename(columns={'AltName': 'Name'}, inplace=True)

def NamesAndIDs(DL): #Function for getting all object IDs in the DataList
    AllIDs = []
    AllNames = []
    for i in range(len(DL)):
        DL[i]["ID"] = DL[i]["ID"].str.replace(" ","")
        AllIDs += (DL[i]["ID"].tolist())
        if "Name" in DL[i].columns:
            DL[i]["Name"] = DL[i]["Name"].str.replace(" ","")
            AllNames += DL[i]["Name"].tolist()
    #print(np.unique(AllIDs))
    UniqueNames = np.unique(AllNames)
    #print(UniqueNames)
    return AllIDs, UniqueNames

def FindOrphanIDs(ID_List): #Function for finding which IDs are only used once (These are likely to be objects that are already in the data but under a differently formatted ID)
    UniqueIDs,Counts = np.unique(ID_List,return_counts=True)
    Orphans = UniqueIDs[Counts==1]
    return Orphans

def FindSimilarIDs(ID_List, Prefix=None, Suffix=None):
    SimilarPrefix = []
    SimilarSuffix = []
    PreLen=len(str(Prefix))
    SufLen=len(str(Suffix))
    for ID in ID_List:
        if (Prefix == None) or (ID[:PreLen] == str(Prefix)):
            SimilarPrefix += [ID]
        if (Suffix == None) or (ID[-SufLen:] == str(Suffix)):
            SimilarSuffix += [ID]
    return np.intersect1d(SimilarSuffix,SimilarPrefix)

def AutoOrphanSolver(ID_List):
    for ID in FindOrphanIDs(ID_List):
        Similars = FindSimilarIDs(ID_List, Prefix=str(ID[0]), Suffix=str(ID[-1]))       
        print("Orphan: "+str(ID))
        print("Similar:"+str(Similars))
        print("----------")

AllIDs, UniqueNames = NamesAndIDs(DL)
#print(FindOrphanIDs(AllIDs))
#print(AutoOrphanSolver(AllIDs))
#print(FindSimilarIDs(UniqueNames, Prefix="A", Suffix="21"))

ID_Aliases = {
    "Palomar12"     :   "Pal12",
    "Ruprecht106"   :   "Rup106",
    "Ter8"          : "Terzan8" }


for i in range(len(DL)):
    DL[i]["ID"]=DL[i]["ID"].replace(ID_Aliases)
    
AllIDs, UniqueNames = NamesAndIDs(DL)
#print(FindOrphanIDs(AllIDs))

[H_P1,H_P2,H_P3,V_T2,K_21] = DL
